In [0]:
from pyspark.sql.functions import col, when

# 1. Load data from the Bronze table instead of a variable
#  Reading the persisted bronze data
df_bronze = spark.table("workspace.bronze.raw_yellow_taxi")

# 2. Create the Silver Schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

# 3. Clean and Cast types
# Performing explicit casting and basic cleaning
df_silver = df_bronze.select(
    col("VendorID").cast("int"),
    
    # Correcting timestamp types for time-based analysis
    col("tpep_pickup_datetime").cast("timestamp"),
    col("tpep_dropoff_datetime").cast("timestamp"),
    
    col("passenger_count").cast("int"),
    col("trip_distance").cast("double"),
    
    # Spatial data casting
    col("pickup_longitude").cast("double"),
    col("pickup_latitude").cast("double"),
    col("dropoff_longitude").cast("double"),
    col("dropoff_latitude").cast("double"),
    
    col("RateCodeID").cast("int"),
    col("store_and_fwd_flag"), 
    col("payment_type").cast("int"),
    
    # Cleaning financial records
    when(col("fare_amount").cast("double") >= 0, col("fare_amount").cast("double")).otherwise(0).alias("fare_amount"),
    col("extra").cast("double"),
    col("mta_tax").cast("double"),
    col("tip_amount").cast("double"),
    col("tolls_amount").cast("double"),
    col("improvement_surcharge").cast("double"),
    col("total_amount").cast("double")
)

# 4. Save to Silver as a Delta Table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.cleaned_yellow_taxi")

# Display the clean schema
df_silver.printSchema()
     